In [1]:
# ============================================================
# CELDA 1
# Librerías necesarias
# ============================================================

import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

In [2]:
# ============================================================
# CELDA 2
# Cargar matriz de estudios incluidos
# ============================================================

archivo_csv = "estudios_primarios.csv"

df = pd.read_csv(
    archivo_csv,
    sep=";",
    encoding="utf-8-sig"
)

print("Total de estudios:", len(df))
print("Columnas disponibles:", len(df.columns))

display(
    df[
        [
            "ID",
            "Título",
            "Método computacional",
            "RQ2"
        ]
    ].head()
)

Total de estudios: 65
Columnas disponibles: 45


,ID,Título,Método computacional,RQ2
0,1,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,"Ontology-Based Data Access (OBDA), población automática de grafos de conocimiento, consultas SPARQL y ejecución de rutinas semánticamente representadas",Parcial
1,2,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,"Enfoque dirigido por modelos basado en Model Driven Architecture (MDA), transformación de modelos y reglas explícitas de correspondencia",Parcial
2,3,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,"Model-driven engineering, transformación modelo?RDF, consultas SPARQL, GraphRAG, recuperación multi-hop y guiada por ontología, GPT-4.1",Parcial
3,4,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support","RAG con embeddings de all-MiniLM-L6-v2, índice FAISS y razonamiento mediante GPT-3.5-turbo; recuperación semántica y síntesis generativa",Sí
4,6,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns,"Generación mediante múltiples LLM; RAG implementado con LangChain, Chroma, embeddings y recuperación top-k; generación Diagram-as-Code",Sí


In [3]:
# ============================================================
# CELDA 3
# Selección de estudios designados directamente a RQ2
#
# Incluidos:
#   - Principal
#   - Sí
#
# Excluidos de la síntesis principal:
#   - Parcial
#   - No aborda
#   - Parcial (adaptación, no seguridad)
#   - Principal (sin seguridad)
# ============================================================

df_rq2 = df[
    df["RQ2"].isin(
        [
            "Principal",
            "Sí"
        ]
    )
].copy()

print(
    "Número de estudios directamente designados a RQ2:",
    len(df_rq2)
)

display(
    df_rq2[
        [
            "ID",
            "Título",
            "Método computacional",
            "RQ2"
        ]
    ].sort_values("ID")
)

Número de estudios directamente designados a RQ2: 25


,ID,Título,Método computacional,RQ2
3,4,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support","RAG con embeddings de all-MiniLM-L6-v2, índice FAISS y razonamiento mediante GPT-3.5-turbo; recuperación semántica y síntesis generativa",Sí
4,6,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns,"Generación mediante múltiples LLM; RAG implementado con LangChain, Chroma, embeddings y recuperación top-k; generación Diagram-as-Code",Sí
6,9,An ontology-based secure design framework for graph-based databases,Razonamiento ontológico; transformación de modelos XMI?RDF; construcción de Knowledge Graph; consultas SPARQL; transformaciones automáticas mediante Python/Docker hacia políticas específicas de DBMS,Sí
14,19,Leveraging LLMs and Knowledge Graphs to Design Secure Automation Systems,LLM + KG; zero-shot class selection; closeness centrality; SPARQL retrieval; triple-to-text templates; cosine similarity,Principal
21,28,Security Requirement Recommendation Method Using Case-Based Reasoning to Prevent Advanced Persistent Threats,Case-Based Reasoning (retrievereusereviseretain); Jaccard similarity + reglas/pesos de similitud; análisis de escenarios; goal/risk reasoning,Principal
22,29,Semanti-MOP: Bridging the semantic gap in automated architecture refactoring via instability-aware Large Language Models,Many-objective evolutionary optimization + instability index/entropy + RAG/LLM semantic arbitration + memetic repair + architectural guardrails,Principal
23,32,"A Comparative Evaluation of RAG Architectures for Cross-Domain LLM Applications: Design, Implementation, and Assessment",LangChain + GPT-4o/LLaMA; sparse/dense/hybrid/fusion retrieval; reranking; RAGAS; bootstrap y pruebas pareadas,Principal
27,38,A Retrieval-Augmented Generation (RAG) System for Supporting Architectural Design of Intelligent Transportation Systems,RAG con retrieval-by-filtering; extracción de metadata mediante LLM; 8 LLMs open-source; Ollama; cuantización 4-bit,Principal
30,43,Design Process for Retrieval Augmented Generation Systems,"Suitability evaluation + proceso sistemático de diseño de retrieval sources, retriever, query generator, reranker, context constructor y generator",Principal
33,46,Enhancing Question-Answering with Knowledge Graph Retrieval and Generation using LLMs,Chunking + embeddings + similarity search + graph traversal + LLM/LangChain,Principal


In [4]:
# ============================================================
# CELDA 4
# Comprobar IDs de RQ2
# ============================================================

ids_rq2 = sorted(df_rq2["ID"].tolist())

print("IDs designados a RQ2:")
print(ids_rq2)

print("\nTotal:", len(ids_rq2))

IDs designados a RQ2:
[4, 6, 9, 19, 28, 29, 32, 38, 43, 46, 49, 52, 53, 55, 58, 60, 67, 68, 69, 71, 75, 76, 78, 79, 81]

Total: 25


In [5]:
# ============================================================
# CELDA 5
# Leer bibliografía LaTeX proporcionada
# ============================================================

archivo_bibliografia = "bibliografia.md"

texto_bib = Path(
    archivo_bibliografia
).read_text(
    encoding="utf-8",
    errors="replace"
)

# Cada entrada comienza con:
# \bibitem{Clave}
patron_bibitem = re.compile(
    r"\\bibitem\{([^}]+)\}\s*"
    r"(.*?)"
    r"(?=\\bibitem\{|\Z)",
    flags=re.DOTALL
)

entradas_bib = []

for clave, bloque in patron_bibitem.findall(texto_bib):

    # Extraer el título contenido en:
    # \textit{*Título*};
    titulo_match = re.search(
        r"\\textit\{\*?(.*?)\*?\};",
        bloque,
        flags=re.DOTALL
    )

    if titulo_match:

        titulo = titulo_match.group(1).strip()

        entradas_bib.append({
            "BibKey": clave,
            "Título_bib": titulo
        })

bibliografia = pd.DataFrame(entradas_bib)

print(
    "Referencias bibliográficas encontradas:",
    len(bibliografia)
)

display(bibliografia.head())

Referencias bibliográficas encontradas: 65


,BibKey,Título_bib
0,Spelten2024,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses
1,Sun2020,A Model Driven Approach to Constructing Knowledge Graph from Relational Database
2,Chis2026,A model-driven knowledge graph treatment for process-centric cyber threat mitigation
3,Khadem2025,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support"
4,Hadjichristofi2026,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns


In [6]:
# ============================================================
# CELDA 6
# Normalización de títulos para realizar el match
# ============================================================

def normalizar_titulo(texto):

    texto = str(texto)

    # Normalizar algunos caracteres frecuentes
    texto = (
        texto
        .replace("", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("-", "-")
    )

    texto = texto.lower()

    # Conservar solamente letras y números
    texto = re.sub(
        r"[^a-z0-9]+",
        " ",
        texto
    )

    return texto.strip()


bibliografia["titulo_normalizado"] = (
    bibliografia["Título_bib"]
    .apply(normalizar_titulo)
)

df_rq2["titulo_normalizado"] = (
    df_rq2["Título"]
    .apply(normalizar_titulo)
)

In [7]:
# ============================================================
# CELDA 7
# Match entre la matriz y los \bibitem
# ============================================================

diccionario_bib = dict(
    zip(
        bibliografia["titulo_normalizado"],
        bibliografia["BibKey"]
    )
)

df_rq2["BibKey"] = (
    df_rq2["titulo_normalizado"]
    .map(diccionario_bib)
)

print(
    "Estudios sin correspondencia bibliográfica:",
    df_rq2["BibKey"].isna().sum()
)

if df_rq2["BibKey"].isna().any():

    display(
        df_rq2[
            df_rq2["BibKey"].isna()
        ][
            [
                "ID",
                "Título"
            ]
        ]
    )

else:

    print(
        "✓ Los 25 estudios tienen correspondencia "
        "con los \\bibitem proporcionados."
    )

Estudios sin correspondencia bibliográfica: 0
✓ Los 25 estudios tienen correspondencia con los \bibitem proporcionados.


In [8]:
# ============================================================
# CELDA 8
# Taxonomía de métodos computacionales para RQ2
#
# Cada estudio se asigna al método computacional dominante.
# Las categorías son mutuamente excluyentes.
# ============================================================

categorias_rq2 = {

    "RAG y recuperación contextual/semántica": [
        4, 6, 32, 38, 43,
        46, 49, 52, 53, 55,
        69, 71, 76, 78, 81
    ],

    "Grafos de conocimiento, ontologías y razonamiento semántico": [
        9, 19, 68, 75, 79
    ],

    "Recuperación guiada por relevancia sobre bases de conocimiento": [
        67
    ],

    "Razonamiento basado en casos (CBR)": [
        28
    ],

    "Optimización evolutiva/memética guiada por LLM/RAG": [
        29
    ],

    "Descubrimiento causal integrado con RAG/LLM": [
        58
    ],

    "Composición dirigida por modelos y refinamiento basado en reglas": [
        60
    ]
}

In [9]:
# ============================================================
# CELDA 9
# Validar la categorización
# ============================================================

ids_categorizados = [
    id_estudio
    for lista_ids in categorias_rq2.values()
    for id_estudio in lista_ids
]

ids_esperados = set(df_rq2["ID"])
ids_obtenidos = set(ids_categorizados)

faltantes = ids_esperados - ids_obtenidos
adicionales = ids_obtenidos - ids_esperados

duplicados = [
    x
    for x in ids_categorizados
    if ids_categorizados.count(x) > 1
]

print("Estudios RQ2:", len(ids_esperados))
print("Estudios categorizados:", len(ids_categorizados))
print("IDs únicos categorizados:", len(ids_obtenidos))

print("\nFaltantes:", sorted(faltantes))
print("Adicionales:", sorted(adicionales))
print("Duplicados:", sorted(set(duplicados)))

if (
    len(faltantes) == 0
    and len(adicionales) == 0
    and len(duplicados) == 0
):
    print(
        "\n✓ La categorización cubre exactamente "
        "los 25 estudios de RQ2."
    )

Estudios RQ2: 25
Estudios categorizados: 25
IDs únicos categorizados: 25

Faltantes: []
Adicionales: []
Duplicados: []

✓ La categorización cubre exactamente los 25 estudios de RQ2.


In [10]:
# ============================================================
# CELDA 10
# Construir tabla de resultados para RQ2
# ============================================================

filas = []

for categoria, ids in categorias_rq2.items():

    estudios = (
        df_rq2[
            df_rq2["ID"].isin(ids)
        ]
        .sort_values("ID")
    )

    claves = estudios["BibKey"].tolist()

    # Construir las citas LaTeX
    citas = (
        "\\cite{"
        + ",".join(claves)
        + "}"
    )

    filas.append({
        "Método computacional": categoria,
        "Estudios designados": citas,
        "Total": len(estudios)
    })


tabla_rq2 = pd.DataFrame(filas)

display(tabla_rq2)

,Método computacional,Estudios designados,Total
0,RAG y recuperación contextual/semántica,"\cite{Khadem2025,Hadjichristofi2026,Ersoy2025,Awadid2025,Xu2025,Gijre2024,Guan2024,Su2025,Omrani2024,Mukherjee2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Thanh2025}",15
1,"Grafos de conocimiento, ontologías y razonamiento semántico","\cite{Paneque2024,Hosseini2025,Gao2021,VergaraVargas2024,Zhang2025b}",5
2,Recuperación guiada por relevancia sobre bases de conocimiento,\cite{Jin2025},1
3,Razonamiento basado en casos (CBR),\cite{Jung2023},1
4,Optimización evolutiva/memética guiada por LLM/RAG,\cite{Salem2026},1
5,Descubrimiento causal integrado con RAG/LLM,\cite{Mascia2025},1
6,Composición dirigida por modelos y refinamiento basado en reglas,\cite{Gstur2024},1


In [11]:
# ============================================================
# CELDA 11
# Añadir total general
# ============================================================

fila_total = pd.DataFrame([
    {
        "Método computacional": "Total",
        "Estudios designados": "",
        "Total": len(df_rq2)
    }
])

tabla_rq2_total = pd.concat(
    [
        tabla_rq2,
        fila_total
    ],
    ignore_index=True
)

display(tabla_rq2_total)

,Método computacional,Estudios designados,Total
0,RAG y recuperación contextual/semántica,"\cite{Khadem2025,Hadjichristofi2026,Ersoy2025,Awadid2025,Xu2025,Gijre2024,Guan2024,Su2025,Omrani2024,Mukherjee2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Thanh2025}",15
1,"Grafos de conocimiento, ontologías y razonamiento semántico","\cite{Paneque2024,Hosseini2025,Gao2021,VergaraVargas2024,Zhang2025b}",5
2,Recuperación guiada por relevancia sobre bases de conocimiento,\cite{Jin2025},1
3,Razonamiento basado en casos (CBR),\cite{Jung2023},1
4,Optimización evolutiva/memética guiada por LLM/RAG,\cite{Salem2026},1
5,Descubrimiento causal integrado con RAG/LLM,\cite{Mascia2025},1
6,Composición dirigida por modelos y refinamiento basado en reglas,\cite{Gstur2024},1
7,Total,,25


In [12]:
# ============================================================
# CELDA 12
# Mostrar categorías y referencias
# ============================================================

for _, fila in tabla_rq2.iterrows():

    print("=" * 80)

    print(
        "Método:",
        fila["Método computacional"]
    )

    print(
        "Total:",
        fila["Total"]
    )

    print(
        "Referencias:",
        fila["Estudios designados"]
    )

Método: RAG y recuperación contextual/semántica
Total: 15
Referencias: \cite{Khadem2025,Hadjichristofi2026,Ersoy2025,Awadid2025,Xu2025,Gijre2024,Guan2024,Su2025,Omrani2024,Mukherjee2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Thanh2025}
Método: Grafos de conocimiento, ontologías y razonamiento semántico
Total: 5
Referencias: \cite{Paneque2024,Hosseini2025,Gao2021,VergaraVargas2024,Zhang2025b}
Método: Recuperación guiada por relevancia sobre bases de conocimiento
Total: 1
Referencias: \cite{Jin2025}
Método: Razonamiento basado en casos (CBR)
Total: 1
Referencias: \cite{Jung2023}
Método: Optimización evolutiva/memética guiada por LLM/RAG
Total: 1
Referencias: \cite{Salem2026}
Método: Descubrimiento causal integrado con RAG/LLM
Total: 1
Referencias: \cite{Mascia2025}
Método: Composición dirigida por modelos y refinamiento basado en reglas
Total: 1
Referencias: \cite{Gstur2024}


In [13]:
# ============================================================
# CELDA 13
# Escapar caracteres especiales
# ============================================================

def escapar_latex(texto):

    texto = str(texto)

    reemplazos = {
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_"
    }

    for original, reemplazo in reemplazos.items():

        texto = texto.replace(
            original,
            reemplazo
        )

    return texto

In [14]:
# ============================================================
# CELDA 14
# Generar tabla LaTeX para RQ2
# ============================================================

def generar_tabla_rq2_latex(tabla):

    lineas = []

    lineas.append(r"\begin{table}[H]")

    lineas.append(
        r"\caption{Métodos computacionales identificados "
        r"en los estudios que responden a RQ2."
        r"\label{tab:rq2-metodos}}"
    )

    lineas.append(
        r"%\isPreprints{\centering}{%"
    )

    lineas.append(
        r"\begin{adjustwidth}{-\extralength}{0cm}"
    )

    lineas.append(r"%}")

    lineas.append(
        r"%\isPreprints{\begin{tabularx}"
        r"{\textwidth}{CCC}}{%"
    )

    lineas.append(
        r"\begin{tabularx}{\fulllength}"
        r"{p{0.30\fulllength} X p{0.08\fulllength}}"
    )

    lineas.append(r"%}")

    lineas.append(r"\toprule")

    lineas.append(
        r"\textbf{Método computacional} & "
        r"\textbf{Estudios designados} & "
        r"\textbf{Total} \\"
    )

    lineas.append(r"\midrule")
    lineas.append("")

    for _, fila in tabla.iterrows():

        metodo = escapar_latex(
            fila["Método computacional"]
        )

        # Las citas NO deben escaparse
        citas = fila["Estudios designados"]

        total = fila["Total"]

        lineas.append(
            f"{metodo} & "
            f"{citas} & "
            f"{total} \\\\"
        )

        lineas.append("")

    lineas.append(r"\midrule")

    lineas.append(
        rf"\textbf{{Total}} & & "
        rf"\textbf{{{len(df_rq2)}}} \\"
    )

    lineas.append(r"\bottomrule")

    lineas.append(r"\end{tabularx}")

    lineas.append(
        r"%\isPreprints{}{%"
    )

    lineas.append(
        r"\end{adjustwidth}"
    )

    lineas.append(r"%}")

    lineas.append(r"\end{table}")

    return "\n".join(lineas)


latex_rq2 = generar_tabla_rq2_latex(
    tabla_rq2
)

print(latex_rq2)

\begin{table}[H]
\caption{Métodos computacionales identificados en los estudios que responden a RQ2.\label{tab:rq2-metodos}}
%\isPreprints{\centering}{%
\begin{adjustwidth}{-\extralength}{0cm}
%}
%\isPreprints{\begin{tabularx}{\textwidth}{CCC}}{%
\begin{tabularx}{\fulllength}{p{0.30\fulllength} X p{0.08\fulllength}}
%}
\toprule
\textbf{Método computacional} & \textbf{Estudios designados} & \textbf{Total} \\
\midrule

RAG y recuperación contextual/semántica & \cite{Khadem2025,Hadjichristofi2026,Ersoy2025,Awadid2025,Xu2025,Gijre2024,Guan2024,Su2025,Omrani2024,Mukherjee2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Thanh2025} & 15 \\

Grafos de conocimiento, ontologías y razonamiento semántico & \cite{Paneque2024,Hosseini2025,Gao2021,VergaraVargas2024,Zhang2025b} & 5 \\

Recuperación guiada por relevancia sobre bases de conocimiento & \cite{Jin2025} & 1 \\

Razonamiento basado en casos (CBR) & \cite{Jung2023} & 1 \\

Optimización evolutiva/memética guiada por LLM/RAG & \cite{Salem202

In [15]:
# ============================================================
# CELDA 15
# Exportar tabla a un archivo LaTeX
# ============================================================

nombre_archivo = "tabla_rq2_metodos.tex"

with open(
    nombre_archivo,
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(latex_rq2)

print(
    f"Archivo generado correctamente: {nombre_archivo}"
)

Archivo generado correctamente: tabla_rq2_metodos.tex


In [16]:
# ============================================================
# CELDA 16
# Crear matriz de trazabilidad de RQ2
# ============================================================

categoria_por_id = {}

for categoria, ids in categorias_rq2.items():

    for id_estudio in ids:

        categoria_por_id[id_estudio] = categoria


df_rq2["Categoría RQ2"] = (
    df_rq2["ID"]
    .map(categoria_por_id)
)

trazabilidad_rq2 = (
    df_rq2[
        [
            "ID",
            "BibKey",
            "Título",
            "Método computacional",
            "Categoría RQ2",
            "RQ2"
        ]
    ]
    .sort_values(
        [
            "Categoría RQ2",
            "ID"
        ]
    )
)

display(trazabilidad_rq2)

trazabilidad_rq2.to_excel(
    "trazabilidad_RQ2.xlsx",
    index=False
)

print(
    "Archivo generado: trazabilidad_RQ2.xlsx"
)

,ID,BibKey,Título,Método computacional,Categoría RQ2,RQ2
45,60,Gstur2024,MoCoRe  A Generic Model-Driven Composition and Rule-Based Refinement Framework,"Model-driven composition + rule-based refinement, framework metamodel-independent",Composición dirigida por modelos y refinamiento basado en reglas,Principal
43,58,Mascia2025,Microservices Performance Testing with Causality-enhanced Large Language Models,CALLMIT: dLiNGAM/causal discovery + RAG + LLM para generar workloads y tests críticos,Descubrimiento causal integrado con RAG/LLM,Principal
6,9,Paneque2024,An ontology-based secure design framework for graph-based databases,Razonamiento ontológico; transformación de modelos XMI?RDF; construcción de Knowledge Graph; consultas SPARQL; transformaciones automáticas mediante Python/Docker hacia políticas específicas de DBMS,"Grafos de conocimiento, ontologías y razonamiento semántico",Sí
14,19,Hosseini2025,Leveraging LLMs and Knowledge Graphs to Design Secure Automation Systems,LLM + KG; zero-shot class selection; closeness centrality; SPARQL retrieval; triple-to-text templates; cosine similarity,"Grafos de conocimiento, ontologías y razonamiento semántico",Principal
50,68,Gao2021,Research on Data Communication Based on Microservice Architecture and Knowledge Graph,Microservice architecture + graph-based knowledge organization/relationship prediction,"Grafos de conocimiento, ontologías y razonamiento semántico",Principal
55,75,VergaraVargas2024,Sarch-Checks: A Method for Checking Software Architecture Security Properties using a Knowledge Graph,Consultas Cypher y checker de seis pasos sobre arquitectura + KG; pattern/tactic conformance,"Grafos de conocimiento, ontologías y razonamiento semántico",Principal
59,79,Zhang2025b,Temporal Knowledge Graph Construction for microservice-based digital power applications,Semantic integration + cross-layer temporal alignment + Kafka event streaming + incremental KG update + graph reasoning,"Grafos de conocimiento, ontologías y razonamiento semántico",Principal
22,29,Salem2026,Semanti-MOP: Bridging the semantic gap in automated architecture refactoring via instability-aware Large Language Models,Many-objective evolutionary optimization + instability index/entropy + RAG/LLM semantic arbitration + memetic repair + architectural guardrails,Optimización evolutiva/memética guiada por LLM/RAG,Principal
3,4,Khadem2025,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support","RAG con embeddings de all-MiniLM-L6-v2, índice FAISS y razonamiento mediante GPT-3.5-turbo; recuperación semántica y síntesis generativa",RAG y recuperación contextual/semántica,Sí
4,6,Hadjichristofi2026,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns,"Generación mediante múltiples LLM; RAG implementado con LangChain, Chroma, embeddings y recuperación top-k; generación Diagram-as-Code",RAG y recuperación contextual/semántica,Sí


Archivo generado: trazabilidad_RQ2.xlsx
